## BPE

### Problem (unicode1):  Understanding Unicode (1 point)

a) a space blanket \
b) printed looks like actual space, it`s __repr__() '\x0' is a representation for space \
c) bellow

In [11]:
print('_' + chr(0) + '_')
'_' + chr(0) + '_'

_ _


'_\x00_'

In [12]:
chr(0).encode('utf-8')

b'\x00'

### Problem (unicode2):  Unicode Encodings (3 points)

a) there are simply less sparsity in bytes lists \
b) cause theare symbols that represented by combination of bytes, for example は -> b'\xe3\x81\xaf', therefore u can`t simply decode b'\xe3' \
c) bytes([227, 129]) - is not correct comb for は, bytes([227, 129, 175]) - correct


In [22]:
test_string = 'hello world!は'
utf8_enc = test_string.encode('utf-8')
utf16_enc = test_string.encode('utf-16')
utf32_enc = test_string.encode('utf-32')

print(utf8_enc)
print(utf16_enc)
print(utf32_enc)

print(len(utf8_enc))
print(len(utf16_enc))
print(len(utf32_enc))

print(list(utf8_enc))
print(list(utf16_enc))
print(list(utf32_enc))

b'hello world!\xe3\x81\xaf'
b'\xff\xfeh\x00e\x00l\x00l\x00o\x00 \x00w\x00o\x00r\x00l\x00d\x00!\x00o0'
b'\xff\xfe\x00\x00h\x00\x00\x00e\x00\x00\x00l\x00\x00\x00l\x00\x00\x00o\x00\x00\x00 \x00\x00\x00w\x00\x00\x00o\x00\x00\x00r\x00\x00\x00l\x00\x00\x00d\x00\x00\x00!\x00\x00\x00o0\x00\x00'
15
28
56
[104, 101, 108, 108, 111, 32, 119, 111, 114, 108, 100, 33, 227, 129, 175]
[255, 254, 104, 0, 101, 0, 108, 0, 108, 0, 111, 0, 32, 0, 119, 0, 111, 0, 114, 0, 108, 0, 100, 0, 33, 0, 111, 48]
[255, 254, 0, 0, 104, 0, 0, 0, 101, 0, 0, 0, 108, 0, 0, 0, 108, 0, 0, 0, 111, 0, 0, 0, 32, 0, 0, 0, 119, 0, 0, 0, 111, 0, 0, 0, 114, 0, 0, 0, 108, 0, 0, 0, 100, 0, 0, 0, 33, 0, 0, 0, 111, 48, 0, 0]


In [35]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    for b in bytestring:
        # print(b)
        print(bytes([b]))
        bytes([b]).decode("utf-8")
    # return "".join([bytes([b]).decode("utf-8") for b in bytestring])

decode_utf8_bytes_to_str_wrong("hello world!は".encode("utf-8"))

b'h'
b'e'
b'l'
b'l'
b'o'
b' '
b'w'
b'o'
b'r'
b'l'
b'd'
b'!'
b'\xe3'


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe3 in position 0: unexpected end of data

In [47]:
bytes([227, 129]).decode('utf-8')

UnicodeDecodeError: 'utf-8' codec can't decode bytes in position 0-1: unexpected end of data

In [49]:
bytes([227, 129, 175]).decode('utf-8')

'は'

### BPE Tokenizer Training

In [2]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

# requires `regex` package
import regex as re
list(re.finditer(PAT, "some, text that i'll pre-tokenize"))

[<regex.Match object; span=(0, 4), match='some'>,
 <regex.Match object; span=(4, 5), match=','>,
 <regex.Match object; span=(5, 10), match=' text'>,
 <regex.Match object; span=(10, 15), match=' that'>,
 <regex.Match object; span=(15, 17), match=' i'>,
 <regex.Match object; span=(17, 20), match="'ll">,
 <regex.Match object; span=(20, 24), match=' pre'>,
 <regex.Match object; span=(24, 25), match='-'>,
 <regex.Match object; span=(25, 33), match='tokenize'>]

In [96]:
# from __future__ import annotations

import os
from collections.abc import Iterable
from typing import IO, Any, BinaryIO

import numpy.typing as npt
import torch
from jaxtyping import Bool, Float, Int
from torch import Tensor

from multiprocessing import Pool, freeze_support
from functools import partial
import regex as re
import itertools
import collections

def find_chunk_boundaries(
    file: BinaryIO,
    desired_num_chunks: int,
    split_special_token: bytes,
) -> list[int]:
    """
    Chunk the file into parts that can be counted independently.
    May return fewer chunks if the boundaries end up overlapping.
    """
    assert isinstance(split_special_token, bytes), "Must represent special token as a bytestring"

    # Get total file size in bytes
    file.seek(0, os.SEEK_END)
    file_size = file.tell()
    file.seek(0)

    chunk_size = file_size // desired_num_chunks

    # Initial guesses for chunk boundary locations, uniformly spaced
    # Chunks start on previous index, don't include last index
    chunk_boundaries = [i * chunk_size for i in range(desired_num_chunks + 1)]
    chunk_boundaries[-1] = file_size

    mini_chunk_size = 4096  # Read ahead by 4k bytes at a time

    for bi in range(1, len(chunk_boundaries) - 1):
        initial_position = chunk_boundaries[bi]
        file.seek(initial_position)  # Start at boundary guess
        while True:
            mini_chunk = file.read(mini_chunk_size)  # Read a mini chunk

            # If EOF, this boundary should be at the end of the file
            if mini_chunk == b"":
                chunk_boundaries[bi] = file_size
                break

            # Find the special token in the mini chunk
            found_at = mini_chunk.find(split_special_token)
            if found_at != -1:
                chunk_boundaries[bi] = initial_position + found_at
                break
            initial_position += mini_chunk_size

    # Make sure all boundaries are unique, but might be fewer than desired_num_chunks
    return sorted(set(chunk_boundaries))

def chunk_pre_tokenize(start: int, end: int, file_path: str, special_tokens: list[str]) -> dict[tuple[bytes, ...], int]:
    with open(file_path, "rb") as f:
        f.seek(start)
        chunk = f.read(end - start).decode("utf-8", errors="ignore")
        special_tokens = "|".join([re.escape(tok) for tok in special_tokens])
        chunks = re.split(special_tokens, chunk)
        chunks = list(filter(('').__ne__, chunks))
        PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
        pre_tokens = {}
        
        for chunk in chunks:    
            for t in re.finditer(PAT, chunk):
                key = tuple(bytes([b]) for b in t.group().encode("utf-8"))
                pre_tokens[key] = pre_tokens.get(key, 0) + 1

    return pre_tokens

def max_rules(freq_dict: dict) -> tuple[bytes, bytes]:
    list_of_pairs = []
    max_freq = max(freq_dict.values())
    for key in freq_dict:
        if freq_dict[key] == max_freq:
            list_of_pairs += [key]

    return max(list_of_pairs)

def merge(pre_tokens: dict, init_vocab: dict, vocab_size=270) -> tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:

    merges_list = []
    temp_freq_pairs = {}
    for tok in pre_tokens:
        for pair in itertools.pairwise(tok):
            if pair in temp_freq_pairs:
                temp_freq_pairs[pair] += pre_tokens[tok]
            else:
                temp_freq_pairs[pair] = pre_tokens[tok]
    
    while len(init_vocab) < vocab_size:
        if not temp_freq_pairs:
            return init_vocab, merges_list # If there are no pairs left
        
        max_token_pair = max_rules(temp_freq_pairs)
        print(max_token_pair)
        init_vocab[len(init_vocab)] = max_token_pair[0] + max_token_pair[1]
        
        tok_list = list(pre_tokens.keys()).copy()
        for tok in tok_list:
            pairs_in_tok = list(itertools.pairwise(list(tok)))
            while max_token_pair in pairs_in_tok:
                idx_max = pairs_in_tok.index(max_token_pair)
                new_key = tuple(
                    list(tok[:idx_max]) + \
                    [max_token_pair[0] + max_token_pair[1]] + \
                    list(tok[idx_max+2:])
                )
                for pair in pairs_in_tok:
                    temp_freq_pairs[pair] -= pre_tokens[tok]
                    if not temp_freq_pairs[pair]:
                        temp_freq_pairs.pop(pair)
                pre_tokens[new_key] = pre_tokens.pop(tok)
                tok = new_key
                pairs_in_tok = list(itertools.pairwise(list(new_key)))
                for pair in pairs_in_tok:
                    if pair in temp_freq_pairs:
                        temp_freq_pairs[pair] += pre_tokens[new_key]
                    else:
                        temp_freq_pairs[pair] = pre_tokens[new_key]
        merges_list.append(max_token_pair)
    
    return temp_freq_pairs, init_vocab, merges_list

def run_train_bpe_test(
    input_path: str | os.PathLike,
    vocab_size: int,
    special_tokens: list[str],
    **kwargs,
) -> tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:
    """Given the path to an input corpus, run train a BPE tokenizer and
    output its vocabulary and merges.

    Args:
        input_path (str | os.PathLike): Path to BPE tokenizer training data.
        vocab_size (int): Total number of items in the tokenizer's vocabulary (including special tokens).
        special_tokens (list[str]): A list of string special tokens to be added to the tokenizer vocabulary.
            These strings will never be split into multiple tokens, and will always be
            kept as a single token. If these special tokens occur in the `input_path`,
            they are treated as any other string.

    Returns:
        tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:
            vocab:
                The trained tokenizer vocabulary, a mapping from int (token ID in the vocabulary)
                to bytes (token bytes)
            merges:
                BPE merges. Each list item is a tuple of bytes (<token1>, <token2>),
                representing that <token1> was merged with <token2>.
                Merges are ordered by order of creation.
    """

    file_path = input_path
    num_processes = kwargs['num_processes'] if 'num_processes' in kwargs else 4
    
    bpe_tokens = {i: tok.encode() for i, tok in enumerate(special_tokens)}
    for i in range(len(special_tokens), 256+len(special_tokens)):
        bpe_tokens[i] = bytes([i-len(special_tokens)]) 
    
    with open(file_path, "rb") as f:
        boundaries = find_chunk_boundaries(
            f, num_processes, b"<|endoftext|>"
        )
    func = partial(
        chunk_pre_tokenize,
        file_path=file_path,
        special_tokens=special_tokens
    )
    results = []
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        results += [func(start, end)]
    res_dict = collections.Counter(results[0])
    if len(results) > 1:
        for i in range(1, len(results)):
            res_dict.update(collections.Counter(results[i]))
    res_dict = dict(res_dict)
    temp_freq_pairs, bpe_tokens, merges = merge(res_dict, bpe_tokens, vocab_size)
    return temp_freq_pairs, bpe_tokens, merges

In [98]:
file_path = 'F:/CS336/HW1/data/test_sample.txt'
temp_freq_pairs, toks, mergs = run_train_bpe_test(file_path, 262, ["<|endoftext|>"])

from tests.adapters import run_train_bpe

toks_real, mergs_real = run_train_bpe(file_path, 262, ["<|endoftext|>"])

print(mergs)
print(mergs_real)

(b' ', b't')
(b'h', b'e')
(b' ', b'a')
(b' ', b's')
(b'n', b'd')
[(b' ', b't'), (b'h', b'e'), (b' ', b'a'), (b' ', b's'), (b'n', b'd')]
[(b' ', b't'), (b'h', b'e'), (b' ', b'a'), (b' ', b's'), (b'n', b'd')]


In [93]:
d = {
    # ('u', 'n', 'c', 'l', 'e'): 1,
    # ('o', 'n', 'c', 'e'): 2,
    # ('n', 'c', 'e'): 2,
    # ('a', 'a', 'a'): 10
}
d = {('a', 'a', 'a', 'a'): 1}
pre_tokens = d
temp_freq_pairs = {}
# cache = {}

for tok in pre_tokens:
    for pair in itertools.pairwise(tok):
        if pair in temp_freq_pairs:
            temp_freq_pairs[pair] += pre_tokens[tok]
        else:
            temp_freq_pairs[pair] = pre_tokens[tok]
    # cache[tok] = list(itertools.pairwise(tok))

for _ in range(10):
    if not temp_freq_pairs:
        break

    print('___POISK PARI_____')
    print(pre_tokens)
    print(temp_freq_pairs)
    max_token_pair = max_rules(temp_freq_pairs)
    print(max_token_pair)

    tok_list = list(pre_tokens.keys()).copy()
    # print(tok_list)
    for tok in tok_list:
        pairs_in_tok = list(itertools.pairwise(list(tok)))
        while max_token_pair in pairs_in_tok:
            print(pairs_in_tok)
            idx_max = pairs_in_tok.index(max_token_pair)
            print(idx_max)
            new_key = tuple(
                list(tok[:idx_max]) + \
                [max_token_pair[0] + max_token_pair[1]] + \
                list(tok[idx_max+2:])
            )
            print(new_key)
            for pair in pairs_in_tok:
                temp_freq_pairs[pair] -= pre_tokens[tok]
                if not temp_freq_pairs[pair]:
                    temp_freq_pairs.pop(pair)
            pre_tokens[new_key] = pre_tokens.pop(tok)
            tok = new_key
            pairs_in_tok = list(itertools.pairwise(list(new_key)))
            for pair in pairs_in_tok:
                if pair in temp_freq_pairs:
                    temp_freq_pairs[pair] += pre_tokens[new_key]
                else:
                    temp_freq_pairs[pair] = pre_tokens[new_key]
    print('___KONEC_____')
    print(pre_tokens)
    print(temp_freq_pairs)
    
# merges_list.append(max_token_pair)


___POISK PARI_____
{('a', 'a', 'a', 'a'): 1}
{('a', 'a'): 3}
('a', 'a')
[('a', 'a'), ('a', 'a'), ('a', 'a')]
0
('aa', 'a', 'a')
[('aa', 'a'), ('a', 'a')]
1
('aa', 'aa')
___KONEC_____
{('aa', 'aa'): 1}
{('aa', 'aa'): 1}
___POISK PARI_____
{('aa', 'aa'): 1}
{('aa', 'aa'): 1}
('aa', 'aa')
[('aa', 'aa')]
0
('aaaa',)
___KONEC_____
{('aaaa',): 1}
{}


In [70]:
pre_tokens

{('u', 'nc', 'l', 'e'): 10, ('o', 'nc', 'e'): 1, ('nc', 'e'): 2}

In [71]:
temp_freq_pairs

{('u', 'n'): 0,
 ('n', 'c'): 0,
 ('c', 'l'): 0,
 ('l', 'e'): 10,
 ('o', 'n'): 0,
 ('c', 'e'): 0,
 ('u', 'nc'): 10,
 ('nc', 'l'): 10,
 ('o', 'nc'): 1,
 ('nc', 'e'): 3}

In [61]:
cache[('u', 'n', 'c', 'l', 'e')]

[('u', 'n'), ('n', 'c'), ('c', 'l'), ('l', 'e')]

In [62]:
[('u', 'nc'), ('nc', 'l'), ('l', 'e')]

[('u', 'nc'), ('nc', 'l'), ('l', 'e')]

In [ ]:
cache[('u'nc', 'c', 'l', 'e')]

In [37]:
toks_real

{0: b'<|endoftext|>',
 1: b'\x00',
 2: b'\x01',
 3: b'\x02',
 4: b'\x03',
 5: b'\x04',
 6: b'\x05',
 7: b'\x06',
 8: b'\x07',
 9: b'\x08',
 10: b'\t',
 11: b'\n',
 12: b'\x0b',
 13: b'\x0c',
 14: b'\r',
 15: b'\x0e',
 16: b'\x0f',
 17: b'\x10',
 18: b'\x11',
 19: b'\x12',
 20: b'\x13',
 21: b'\x14',
 22: b'\x15',
 23: b'\x16',
 24: b'\x17',
 25: b'\x18',
 26: b'\x19',
 27: b'\x1a',
 28: b'\x1b',
 29: b'\x1c',
 30: b'\x1d',
 31: b'\x1e',
 32: b'\x1f',
 33: b' ',
 34: b'!',
 35: b'"',
 36: b'#',
 37: b'$',
 38: b'%',
 39: b'&',
 40: b"'",
 41: b'(',
 42: b')',
 43: b'*',
 44: b'+',
 45: b',',
 46: b'-',
 47: b'.',
 48: b'/',
 49: b'0',
 50: b'1',
 51: b'2',
 52: b'3',
 53: b'4',
 54: b'5',
 55: b'6',
 56: b'7',
 57: b'8',
 58: b'9',
 59: b':',
 60: b';',
 61: b'<',
 62: b'=',
 63: b'>',
 64: b'?',
 65: b'@',
 66: b'A',
 67: b'B',
 68: b'C',
 69: b'D',
 70: b'E',
 71: b'F',
 72: b'G',
 73: b'H',
 74: b'I',
 75: b'J',
 76: b'K',
 77: b'L',
 78: b'M',
 79: b'N',
 80: b'O',
 81: b'P',
 82: b

In [22]:
mergs

[(b' ', b't'),
 (b'h', b'e'),
 (b't', b'h'),
 (b' ', b'a'),
 (b'n', b'd'),
 (b'a', b'n'),
 (b' ', b'w'),
 (b'e', b'd'),
 (b' ', b's'),
 (b'e', b'r'),
 (b'r', b'e'),
 (b' t', b'h'),
 (b't', b'o'),
 (b' t', b'he'),
 (b' t', b'o'),
 (b'i', b'n'),
 (b'a', b'r'),
 (b' ', b'T'),
 (b' ', b'h'),
 (b'o', b'm'),
 (b' a', b'n'),
 (b' ', b'b'),
 (b'he', b'y')]

In [21]:
toks

{0: b'<|endoftext|>',
 1: b'\x00',
 2: b'\x01',
 3: b'\x02',
 4: b'\x03',
 5: b'\x04',
 6: b'\x05',
 7: b'\x06',
 8: b'\x07',
 9: b'\x08',
 10: b'\t',
 11: b'\n',
 12: b'\x0b',
 13: b'\x0c',
 14: b'\r',
 15: b'\x0e',
 16: b'\x0f',
 17: b'\x10',
 18: b'\x11',
 19: b'\x12',
 20: b'\x13',
 21: b'\x14',
 22: b'\x15',
 23: b'\x16',
 24: b'\x17',
 25: b'\x18',
 26: b'\x19',
 27: b'\x1a',
 28: b'\x1b',
 29: b'\x1c',
 30: b'\x1d',
 31: b'\x1e',
 32: b'\x1f',
 33: b' ',
 34: b'!',
 35: b'"',
 36: b'#',
 37: b'$',
 38: b'%',
 39: b'&',
 40: b"'",
 41: b'(',
 42: b')',
 43: b'*',
 44: b'+',
 45: b',',
 46: b'-',
 47: b'.',
 48: b'/',
 49: b'0',
 50: b'1',
 51: b'2',
 52: b'3',
 53: b'4',
 54: b'5',
 55: b'6',
 56: b'7',
 57: b'8',
 58: b'9',
 59: b':',
 60: b';',
 61: b'<',
 62: b'=',
 63: b'>',
 64: b'?',
 65: b'@',
 66: b'A',
 67: b'B',
 68: b'C',
 69: b'D',
 70: b'E',
 71: b'F',
 72: b'G',
 73: b'H',
 74: b'I',
 75: b'J',
 76: b'K',
 77: b'L',
 78: b'M',
 79: b'N',
 80: b'O',
 81: b'P',
 82: b

In [9]:
special_tokens = ["<|endoftext|>"]
num_processes = 4
bpe_tokens = {i: tok.encode() for i, tok in enumerate(special_tokens)}
for i in range(len(special_tokens), 256+len(special_tokens)):
    bpe_tokens[i] = bytes([i-len(special_tokens)]) 

with open(file_path, "rb") as f:
    boundaries = find_chunk_boundaries(
        f, num_processes, b"<|endoftext|>"
    )
func = partial(
    chunk_pre_tokenize,
    file_path=file_path,
    special_tokens=special_tokens
)
results = []
for start, end in zip(boundaries[:-1], boundaries[1:]):
    results += [func(start, end)]
res_dict = collections.Counter(results[0])
if len(results) > 1:
    for i in range(1, len(results)):
        res_dict.update(collections.Counter(results[i]))

In [16]:
merges_list = []
temp_freq_pairs = {}
for tok in res_dict:
    for pair in itertools.pairwise(tok):
        if pair in temp_freq_pairs:
            temp_freq_pairs[pair] += res_dict[tok]
        else:
            temp_freq_pairs[pair] = res_dict[tok]

In [ ]:
temp_freq_pairs[(b' ', b't')]

144